In [ ]:
%%capture
!pip install unsloth
# 同时获取最新的版本 Unsloth！
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
!pip install --upgrade transformers torch peft

  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 93.1 MB/s eta 0:00:00
Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl (207.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 31.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.51.1
    Uninstalling transformers-4.51.1:
      Successfully uninstalled transformers-4.51.1
  Attempting uninstall: peft
    Found existing installation: peft 0.14.0
    Uninstalling peft-0.14.0:
      Successfully uninstalled peft-0.14.0


In [ ]:
# 导入 Unsloth 库中的 FastLanguageModel 类
import unsloth
from unsloth import FastLanguageModel
import torch

# 设置模型输入序列的最大长度，单位为 token。这个值限制了每次模型处理的文本长度
max_seq_length = 256

# 设置模型的数据类型，如果为 None，通常会默认使用 float32
dtype = torch.bfloat16

# 设置是否以 4-bit 精度加载模型。设置为 True 可以减少内存占用和计算量，但可能会降低精度
load_in_4bit = False

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Failed to patch Gemma3ForConditionalGeneration.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
## 使用本地环境
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HUGGINGFACE_TOKEN')
login(hf_token)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=max_seq_length,
    dtype=torch.bfloat16,  # A100 推荐 bf16
    load_in_4bit=False,    # A100 建议全精度或 bf16
    token=hf_token,
)

==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [ ]:
prompt_style_1 = """### 指令: 从儿童叙事文本中提取标准化叙事事件。严格按照格式输出。

**事件结构:**
(触发词；主语；宾语；时间状语；地点状语)
缺失信息用“无”，多个主语或宾语用逗号分隔。

**示例:**
输入：小男孩一不小心从树上掉了下来.
输出：(掉；小男孩；无；无；从树上)

### 输入文本:
{input}

### 输出: """

In [ ]:
# 中文问答问题
question = """他们在找小青蛙."""

# 构造用户输入
user_input = question.strip()

# 根据提示模板和问题构造输入
inputs = tokenizer([prompt_style_1.format(input=user_input)], return_tensors="pt").to("cuda")

# 启动快速推理
FastLanguageModel.for_inference(model)  # Unsloth 已实现2倍加速推理！

# 模型生成答案
outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=128,
    use_cache=True,
)

In [ ]:
# 解码输出并提取回答内容
response = tokenizer.batch_decode(outputs)
print(response[0].split("### 输出: ")[1].strip())

(无；他们；小青蛙；无；无) ###

### 输入文本:
他妈妈带我去公园玩。


In [ ]:
from datasets import load_dataset
dataset=load_dataset("json", data_files="/content/output_train.jsonl")
print(dataset)
print("数据集的字段：", dataset.column_names)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'event'],
        num_rows: 12540
    })
})
数据集的字段： {'train': ['text', 'event']}


In [ ]:
train_dataset = dataset['train']
# 查看第一个生成的文本
# print(dataset["text"][0])
print(train_dataset)

Dataset({
    features: ['text', 'event'],
    num_rows: 12540
})


In [ ]:
print(train_dataset[0]["text"])

青蛙在瓶子里的时候.


In [ ]:
prompt="""### 指令: 从儿童叙事文本中提取标准化叙事事件。严格按照格式输出。

**事件结构:**
(触发词；主语；宾语；时间状语；地点状语)
缺失信息用“无”，多个主语或宾语用逗号分隔。

**示例:**
输入：小男孩一不小心从树上掉了下来
输出：(掉；小男孩；无；无；从树上)

输入：第二天早上小狗和小男孩发现瓶子里的青蛙不见了.
输出：(发现；小狗，小男孩；瓶子里的青蛙不见了；第二天早上；无)

### 输入文本:{input}

### 输出:{output}"""
EOS_TOKEN = tokenizer.eos_token

In [ ]:
def formatting_prompts_func(examples):  # Takes a batch of dataset examples as input
    inputs = examples["text"]       # Extracts the medical question from the dataset
    outputs = examples["event"]

    texts = []
    for input_text, output_text in zip(inputs, outputs):
        prompt_template = prompt
        text = prompt_template.format(input=input_text, output=output_text) + EOS_TOKEN
        texts.append(text)

    # Shuffle
    combined = list(zip(texts, outputs))
    # random.shuffle(combined)
    texts, outputs = zip(*combined)

    return {"text": list(texts)}

In [ ]:
dataset_finetune = train_dataset.map(formatting_prompts_func, batched = True)
dataset_finetune["text"][1]

Map:   0%|          | 0/12540 [00:00<?, ? examples/s]

'### 指令: 从儿童叙事文本中提取标准化叙事事件。严格按照格式输出。\n\n**事件结构:**\n(触发词；主语；宾语；时间状语；地点状语)\n缺失信息用“无”，多个主语或宾语用逗号分隔。\n\n**示例:**\n输入：小男孩一不小心从树上掉了下来\n输出：(掉；小男孩；无；无；从树上)\n\n输入：第二天早上小狗和小男孩发现瓶子里的青蛙不见了.\n输出：(发现；小狗，小男孩；瓶子里的青蛙不见了；第二天早上；无)\n\n### 输入文本:他就睡了会儿觉.\n\n### 输出:(睡；他；觉；无；无)<|im_end|>'

In [ ]:
dataset_dev=load_dataset("json", data_files="/content/output_dev.jsonl")
print(dataset_dev)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'event'],
        num_rows: 3511
    })
})


In [ ]:
dev_dataset=dataset_dev['train']
print(dev_dataset[0])

{'text': '他们两个睡觉了.', 'event': '(睡觉；他们两个；无；无；无)'}


In [ ]:
dataset_finetune_dev = dev_dataset.map(formatting_prompts_func, batched = True)
dataset_finetune_dev["text"][2]

Map:   0%|          | 0/3511 [00:00<?, ? examples/s]

'### 指令: 从儿童叙事文本中提取标准化叙事事件。严格按照格式输出。\n\n**事件结构:**\n(触发词；主语；宾语；时间状语；地点状语)\n缺失信息用“无”，多个主语或宾语用逗号分隔。\n\n**示例:**\n输入：小男孩一不小心从树上掉了下来\n输出：(掉；小男孩；无；无；从树上)\n\n输入：第二天早上小狗和小男孩发现瓶子里的青蛙不见了.\n输出：(发现；小狗，小男孩；瓶子里的青蛙不见了；第二天早上；无)\n\n### 输入文本:小青蛙不见了.7\n\n### 输出:(不见；小青蛙；无；无；无)<|im_end|>'

In [ ]:
dataset_finetune_mini=dataset_finetune.select(range(1000))
dataset_finetune_mini["text"][0]

'### 指令: 从儿童叙事文本中提取标准化叙事事件。严格按照格式输出。\n\n**事件结构:**\n(触发词；主语；宾语；时间状语；地点状语)\n缺失信息用“无”，多个主语或宾语用逗号分隔。\n\n**示例:**\n输入：小男孩一不小心从树上掉了下来\n输出：(掉；小男孩；无；无；从树上)\n\n输入：第二天早上小狗和小男孩发现瓶子里的青蛙不见了.\n输出：(发现；小狗，小男孩；瓶子里的青蛙不见了；第二天早上；无)\n\n### 输入文本:青蛙在瓶子里的时候.\n\n### 输出:(在；青蛙；瓶子里；无；无)<|im_end|>'

In [ ]:
model_lora = FastLanguageModel.get_peft_model(
    model=model,  # 待微调的模型
    r=16,  # LoRA 分解的秩，保持为 8，适合大型模型和大数据集
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        # 仅对注意力头的投影层应用 LoRA，符合 Qwen 模型架构
    ],
    lora_alpha=32,  # 调整为 8，与 r 匹配，结合 RSLoRA 稳定训练
    lora_dropout=0.1,  # 保持 0.1，防止过拟合，适合大数据集
    bias="lora_only",  # 不修改偏置项，保持默认设置
    use_gradient_checkpointing=True,  # 启用梯度检查点，节省显存，适合 32B 模型
    random_state=3407,  # 固定随机种子，确保训练可复现
    use_rslora=True,  # 启用 RSLoRA，提升训练稳定性
    loftq_config=None,  # 保持示例配置，可根据需求调整
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth: bias = `none` is supported for fast patching. You are using bias = lora_only.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.3.19 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported, FastLanguageModel
import wandb

In [ ]:
trainer = SFTTrainer(
    model=model_lora,  # The model to be fine-tuned
    tokenizer=tokenizer,  # Tokenizer to process text inputs
    train_dataset=dataset_finetune,  # Dataset used for training
    eval_dataset=dataset_finetune_dev,  # Dataset used for evaluation (optional)
    dataset_text_field="text",  # Specifies which field in the dataset contains training text
    max_seq_length=max_seq_length,  # Defines the maximum sequence length for inputs
    dataset_num_proc=4,  # Uses 2 CPU threads to speed up data preprocessing

    # Define training arguments
    args=TrainingArguments(
        per_device_train_batch_size=64,  # Number of examples processed per device (GPU) at a time
        gradient_accumulation_steps=4,  # Accumulate gradients over 4 steps before updating weights
        num_train_epochs=10, # Full fine-tuning run
        warmup_ratio=0.06,  # Gradually increases learning rate for the first 5 steps
        # max_steps=50,  # Limits training to 60 steps (useful for debugging; increase for full fine-tuning)
        learning_rate=5e-6,  # Learning rate for weight updates (tuned for LoRA fine-tuning)
        max_grad_norm=0.8,
        # fp16=not is_bfloat16_supported(),  # Use FP16 (if BF16 is not supported) to speed up training
        # bf16=is_bfloat16_supported(),  # Use BF16 if supported (better numerical stability on newer GPUs)
        bf16=True,
        logging_steps=10,  # Logs training progress every 10 steps
        optim="adamw_torch_fused",  # Uses memory-efficient AdamW optimizer in 8-bit mode
        weight_decay=0.05,  # Regularization to prevent overfitting
        lr_scheduler_type="cosine",  # Uses a linear learning rate schedule
        seed=3704,  # Sets a fixed seed for reproducibility
        output_dir="/content/outputs",  # Directory where fine-tuned model checkpoints will be saved

        eval_strategy="steps",      # 启用按步骤评估
        eval_steps=20,             # 每 50 步评估一次
        per_device_eval_batch_size=256,      # 验证批次大小
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/12540 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/3511 [00:00<?, ? examples/s]

In [ ]:
wnb_token=userdata.get('wandb_token')

In [ ]:
wandb.login(key=wnb_token) # import wandb
run = wandb.init(
    project='test0418_qwen7B_event_SFT',
    entity='FeSCN',
    job_type="training",
    settings=wandb.Settings(init_timeout=120),
    anonymous="allow"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yuxuan0612 (FeSCN) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,540 | Num Epochs = 5 | Total steps = 245
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 4 x 1) = 256
 "-____-"     Trainable parameters = 6,881,280/7,622,497,792 (0.09% trained)


Step,Training Loss,Validation Loss
20,2.502400,2.431307
40,2.274600,2.206269
60,2.040400,1.966027
80,1.780600,1.702546
100,1.504100,1.427970
120,1.238200,1.167767
140,1.006400,0.947499
160,0.834700,0.794303
180,0.734300,0.711462
200,0.683300,0.670136


Step,Training Loss,Validation Loss
20,2.502400,2.431307
40,2.274600,2.206269
60,2.040400,1.966027
80,1.780600,1.702546
100,1.504100,1.427970
120,1.238200,1.167767
140,1.006400,0.947499
160,0.834700,0.794303
180,0.734300,0.711462
200,0.683300,0.670136


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,540 | Num Epochs = 10 | Total steps = 490
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 4 x 1) = 256
 "-____-"     Trainable parameters = 6,881,280/7,622,497,792 (0.09% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
20,2.880400,2.825058
40,2.522300,2.410587
60,2.162100,2.067382
80,1.834700,1.741668
100,1.491800,1.387960
120,1.158800,1.071167
140,0.867200,0.797763
160,0.666700,0.617026
180,0.529200,0.488851
200,0.411200,0.382874


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss,Validation Loss
20,2.880400,2.825058
40,2.522300,2.410587
60,2.162100,2.067382
80,1.834700,1.741668
100,1.491800,1.387960
120,1.158800,1.071167
140,0.867200,0.797763
160,0.666700,0.617026
180,0.529200,0.488851
200,0.411200,0.382874


In [ ]:
prompt="""### 指令: 从儿童叙事文本中提取标准化叙事事件。严格按照格式输出。

**事件结构:**
(触发词；主语；宾语；时间状语；地点状语)
缺失信息用“无”，多个主语或宾语用逗号分隔。

**示例:**
输入：小男孩一不小心从树上掉了下来
输出：(掉；小男孩；无；无；从树上)

输入：第二天早上小狗和小男孩发现瓶子里的青蛙不见了.
输出：(发现；小狗，小男孩；瓶子里的青蛙不见了；第二天早上；无)

### 输入文本:{input}

### 输出:"""

In [ ]:
question = """那个蜜蜂在蜂窝里飞了出来."""

# Load the inference model using FastLanguageModel (Unsloth optimizes for speed)
FastLanguageModel.for_inference(model_lora)  # Unsloth has 2x faster inference!

# Tokenize the input question with a specific prompt format and move it to the GPU
inputs = tokenizer([prompt.format(input=question)], return_tensors="pt").to("cuda")

# Generate a response using LoRA fine-tuned model with specific parameters
outputs = model_lora.generate(
    input_ids=inputs.input_ids,          # Tokenized input IDs
    attention_mask=inputs.attention_mask, # Attention mask for padding handling
    max_new_tokens=128,                  # Maximum length for generated response
    use_cache=True,                        # Enable cache for efficient generation
)

# Decode the generated response from tokenized format to readable text
response = tokenizer.batch_decode(outputs)

# Extract and print only the model's response part after "### Response:"
print(response[0])

### 指令: 从儿童叙事文本中提取标准化叙事事件。严格按照格式输出。
先分析句子结构，识别核心谓语动词（触发词），再匹配施事（主语）、受事（宾语），最后提取时空成分。!!务必严格遵循以下格式!!

**事件结构:**
(触发词；主语；宾语；时间状语；地点状语)
缺失信息用“无”，多个主语或宾语用逗号分隔。

**示例:**
输入：小男孩一不小心从树上掉了下来
输出：(掉；小男孩；无；无；从树上)

输入：第二天早上小狗和小男孩发现瓶子里的青蛙不见了.
输出：(发现；小狗，小男孩；瓶子里的青蛙不见了；第二天早上；无)

### 输入文本:那个蜜蜂在蜂窝里飞了出来.

### 输出: (飞；蜜蜂；蜂窝；无；无)### 事件结构:
(触发词；主语；宾语；时间状语；地点状语)
(飞；蜜蜂；蜂窝；无；无)###<|endoftext|>


In [ ]:
wandb.finish()

eval/loss,█▇▆▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/runtime,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/samples_per_second,▁███████████████████████
eval/steps_per_second,▁█████▇▇█▇▇█▇▇█▇▇▇▇█▇▇▇▇
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇█████
train/global_step,▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇█████
train/grad_norm,██▆▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▃▅████████▇▇▇▇▇▆▆▆▅▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
train/loss,███▇▆▆▅▄▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.27183
eval/runtime,48.5683


In [ ]:
from tqdm import tqdm
import os
import json

In [ ]:
from google.colab import drive


In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
drive_output="/content/drive/MyDrive/two stage results/stage one"

In [ ]:
FastLanguageModel.for_inference(model_lora)

jsonl_path="/content/output_eval.jsonl"
output_jsonl = "/content/drive/MyDrive/two stage results/stage one/finetune_result_Qwen7B_10epoch_0418_1.jsonl"
results = []

In [ ]:
with open(jsonl_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

for idx, line in enumerate(tqdm(lines, desc="生成中", unit="行"), start=1):
    data = json.loads(line)
    question = data.get("text", "").strip()

    # 构造输入
    inputs = tokenizer([prompt.format(input=question)], return_tensors="pt").to("cuda")

    # 推理生成
    outputs = model_lora.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=128,
        use_cache=True,
    )

    # 解码
    response = tokenizer.batch_decode(outputs)[0]
    answer = response.split("### 输出:")[1].strip() if "### 输出:" in response else response.strip()

    # 保存到内存列表
    results.append({
        "line_id": idx,
        "question": question,
        "answer": answer
    })

# 统一写入一个 jsonl 文件
with open(output_jsonl, 'w', encoding='utf-8') as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"全部完成！答案已保存为 {output_jsonl}")

生成中: 100%|██████████| 1926/1926 [20:26<00:00,  1.57行/s]

全部完成！答案已保存为 /content/drive/MyDrive/two stage results/stage one/finetune_result_Qwen7B_10epoch_0418_1.jsonl


In [ ]:
from huggingface_hub import HfApi

from huggingface_hub import create_repo

repo_name = "Venassa/Qwen2.5-7B-Children_Narrative_Extraction-event_extraction_version_10epoch0418"
create_repo(repo_name, exist_ok=True)

# push 上传
model_lora.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Saved model to https://huggingface.co/Venassa/Qwen2.5-7B-Children_Narrative_Extraction-event_extraction_version_10epoch0418


README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]